# Jina OCR sample — Elastic Inference Service (EIS)

This notebook extracts the text from an image (`open-jar.jpeg`) using the **Jina OCR** model served by **Elastic Inference Service (EIS)**.

**How to use:** fill in **Step 2** only, then run the cells in order with **Shift+Enter**.

## Step 1 — Install

In [ ]:
%pip install -q requests

## Step 2 — ✏️ Fill in your settings

- `ES_URL`: your Elasticsearch URL
- `ES_API_KEY`: your API key
- `INFERENCE_ID`: the Jina OCR endpoint ID (run `GET _inference` in Kibana to find it)
- `IMAGE_PATH`: the image to read (the sample image is in the same folder as this notebook)

In [ ]:
ES_URL       = "https://elastic-endpoint.es.us-east-1.aws.found.io"
ES_API_KEY   = "YOUR_API_KEY_HERE"
INFERENCE_ID = ".jina-ocr-v1-chat_completion"
IMAGE_PATH   = "open-jar.jpeg"

## Step 3 — Send the image to Jina OCR

The image is encoded as **base64** and sent together with a short instruction.
The answer arrives in pieces (*streaming*), so we join the pieces into a single text.

In [ ]:
import base64, json, requests

# 1. Read the image and encode it as base64
with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

# 2. Build the request: an instruction + the image
body = {
    "messages": [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Extract all the text from this image. Keep the structure (titles, steps and tables) as raw text."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
        ],
    }]
}

# 3. Call the EIS inference endpoint
url = f"{ES_URL.rstrip('/')}/_inference/chat_completion/{INFERENCE_ID}/_stream"
response = requests.post(url, json=body, headers={"Authorization": f"ApiKey {ES_API_KEY}"}, stream=True)
response.raise_for_status()

# 4. Join the streamed pieces into the final text
text = ""
for line in response.iter_lines(decode_unicode=True):
    if line.startswith("data:") and "[DONE]" not in line:
        for choice in json.loads(line[5:])["choices"]:
            text += choice["delta"].get("content") or ""

## Step 4 — Result

The image we sent and, right below it, the text extracted by the model.

In [ ]:
from IPython.display import Image, display

display(Image(filename=IMAGE_PATH, width=450))

print(text)